In [1]:
import pandas as pd
import numpy as np
import os
import time
from azure.core.credentials import AzureKeyCredential
from azure.ai.textanalytics import TextAnalyticsClient
from sklearn.metrics import accuracy_score, classification_report
from dotenv import load_dotenv
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import re
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from gensim.models import Word2Vec

In [3]:
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')

df_reviews = pd.read_csv('C:/Laborator AI/Laborator 8/reviews_mixed.csv')
df_curat = df_reviews.dropna(subset=['Text', 'Sentiment']).reset_index(drop=True)

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def curata_text(text):
    text = str(text).lower()

    text = re.sub(r"won't", "will not", text)
    text = re.sub(r"can\'t", "can not", text)
    text = re.sub(r"n\'t", " not", text)
    text = re.sub(r"\'re", " are", text)
    text = re.sub(r"\'s", " is", text)
    text = re.sub(r"\'d", " would", text)
    text = re.sub(r"\'ll", " will", text)
    text = re.sub(r"\'t", " not", text)
    text = re.sub(r"\'ve", " have", text)
    text = re.sub(r"\'m", " am", text)

    text = re.sub(r'[^a-z\s]', ' ', text)

    cuvinte = word_tokenize(text)

    cuvinte_curate = [lemmatizer.lemmatize(cuvant) for cuvant in cuvinte if cuvant not in stop_words]

    return ' '.join(cuvinte_curate)

print("Curățăm datele din setul original...")
df_curat['Procesat'] = df_curat['Text'].astype(str).apply(curata_text)

mesaje_test = df_curat['Procesat'].tolist()
print("Preprocesare finalizată!")

Curățăm datele din setul original...
Preprocesare finalizată!


[nltk_data] Downloading package punkt to C:\Users\Ciocan
[nltk_data]     Ionut\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Ciocan
[nltk_data]     Ionut\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Ciocan
[nltk_data]     Ionut\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Ciocan
[nltk_data]     Ionut\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
load_dotenv(override=True)

AZURE_KEY = os.getenv("AZURE_API_KEY")
AZURE_ENDPOINT = os.getenv("AZURE_ENDPOINT")

COLOANA_TEXT = 'Text'
COLOANA_ETICHETA = 'Sentiment'

def evalueaza_tot_setul(df, text_col, label_col, batch_size=10, delay_secunde=3.1):
    """
    Trimite textele către Azure în loturi, aplică delay și calculează acuratețea.
    (Pauza de 3.1 secunde între apeluri asigură sub 20 de apeluri pe minut).
    """
    try:
        credential = AzureKeyCredential(AZURE_KEY)
        client = TextAnalyticsClient(endpoint=AZURE_ENDPOINT, credential=credential)
    except Exception as e:
        print(f"Eroare la inițializarea clientului Azure: {e}")
        return None

    texte = df[text_col].astype(str).tolist()
    etichete_reale = df[label_col].astype(str).str.lower().tolist()

    predictii_azure = []

    print(f"Începem procesarea a {len(texte)} texte în loturi de {batch_size}...")

    for i in range(0, len(texte), batch_size):
        lot_texte = texte[i : i + batch_size]

        try:
            response = client.analyze_sentiment(documents=lot_texte)
            for doc in response:
                if not doc.is_error:
                    sentiment = doc.sentiment.lower()
                    if sentiment in ['neutral', 'mixed']:
                        scor_pozitiv = doc.confidence_scores.positive
                        scor_negativ = doc.confidence_scores.negative

                        if scor_pozitiv > scor_negativ:
                            predictii_azure.append('positive')
                        else:
                            predictii_azure.append('negative')
                    else:
                        predictii_azure.append(sentiment)
                else:
                    print(f"Eroare la un document din lotul {i}: {doc.error.message}")
                    predictii_azure.append("eroare")

        except Exception as e:
            print(f"Eroare la apelul Azure pentru lotul {i}: {e}")
            predictii_azure.extend(['eroare'] * len(lot_texte))

        print(f"Procesate: {min(i + batch_size, len(texte))}/{len(texte)}")

        if i + batch_size < len(texte):
            time.sleep(delay_secunde)

    # Filtrăm textele care au dat eroare la API pentru a nu strica statistica
    indici_valizi = [i for i, p in enumerate(predictii_azure) if p != 'eroare']
    y_true = [etichete_reale[i] for i in indici_valizi]
    y_pred = [predictii_azure[i] for i in indici_valizi]

    acuratete = accuracy_score(y_true, y_pred)
    print("="*40)
    print(f"ACURATEȚE GENERALĂ AZURE: {acuratete * 100:.2f}%")
    print("="*40)

    df_rezultat = df.iloc[indici_valizi].copy()
    df_rezultat['Predictie_Azure'] = y_pred
    df_rezultat[label_col] = [eticheta for eticheta in y_true]

    return df_rezultat

df_rezultate_finale = evalueaza_tot_setul(df_curat, COLOANA_TEXT, COLOANA_ETICHETA)

Începem procesarea a 207 texte în loturi de 10...
Procesate: 10/207
Procesate: 20/207
Procesate: 30/207
Procesate: 40/207
Procesate: 50/207
Procesate: 60/207
Procesate: 70/207
Procesate: 80/207
Procesate: 90/207
Procesate: 100/207
Procesate: 110/207
Procesate: 120/207
Procesate: 130/207
Procesate: 140/207
Procesate: 150/207
Procesate: 160/207
Procesate: 170/207
Procesate: 180/207
Procesate: 190/207
Procesate: 200/207
Procesate: 207/207
ACURATEȚE GENERALĂ AZURE: 90.82%


In [6]:
if df_rezultate_finale is not None:
    print("\n--- 1. MATRICEA DE CONFUZIE ---")
    matrice_confuzie = pd.crosstab(df_rezultate_finale[COLOANA_ETICHETA],
                                   df_rezultate_finale['Predictie_Azure'],
                                   rownames=['Real (CSV)'],
                                   colnames=['Predictie (Azure)'])
    display(matrice_confuzie)

    print("\n--- 2. EXTRAGEREA AUTOMATĂ A GREȘELILOR ---")
    df_greseli = df_rezultate_finale[df_rezultate_finale[COLOANA_ETICHETA] != df_rezultate_finale['Predictie_Azure']]

    print(f"Din totalul de texte, Azure a clasificat diferit față de CSV un număr de {len(df_greseli)} texte.")

    display(df_greseli[[COLOANA_TEXT, COLOANA_ETICHETA, 'Predictie_Azure']])


--- 1. MATRICEA DE CONFUZIE ---


Predictie (Azure),negative,positive
Real (CSV),,
negative,128,14
positive,5,60



--- 2. EXTRAGEREA AUTOMATĂ A GREȘELILOR ---
Din totalul de texte, Azure a clasificat diferit față de CSV un număr de 19 texte.


,Text,Sentiment,Predictie_Azure
12,So if you're the type that likes to let water ...,negative,positive
19,very light and the comfy of the bed was unbeat...,negative,positive
22,Microwave needed!,negative,positive
27,"The building was under renovation,",positive,negative
35,The walk up 4 stories,negative,positive
51,Outside seating was lovely,negative,positive
54,Bathroom area large would have been nice with ...,positive,negative
75,Roof terrace great,negative,positive
77,the room had aircon and we had earplugs and sl...,positive,negative
90,Wifi connected,negative,positive


In [7]:
print("=== 1. BAG OF WORDS (BoW) ===")
# Setăm max_features=500 pentru a păstra doar cele mai importante 500 de cuvinte
# (altfel, matricea ar deveni uriașă și ar încetini antrenarea)
vectorizer_bow = CountVectorizer(max_features=500, stop_words='english')
X_bow = vectorizer_bow.fit_transform(mesaje_test)

# Creăm un DataFrame doar pentru a vizualiza rezultatul
df_bow = pd.DataFrame(X_bow.toarray(), columns=vectorizer_bow.get_feature_names_out())
print(f"Forma matricii BoW: {df_bow.shape} (texte x cuvinte)")
display(df_bow.head(3))


print("\n=== 2. TF-IDF ===")
vectorizer_tfidf = TfidfVectorizer(max_features=500, stop_words='english')
X_tfidf = vectorizer_tfidf.fit_transform(mesaje_test)

df_tfidf = pd.DataFrame(X_tfidf.toarray(), columns=vectorizer_tfidf.get_feature_names_out())
print(f"Forma matricii TF-IDF: {df_tfidf.shape} (texte x cuvinte)")
display(df_tfidf.head(3))


print("=== 3. WORD2VEC ===")

# 1. Tokenizarea: spargem propozițiile în liste de cuvinte
texte_tokenizate = [word_tokenize(text.lower()) for text in mesaje_test]

# 2. Antrenăm modelul Word2Vec pe textele noastre
# vector_size = 50 înseamnă că fiecare cuvânt va fi reprezentat de 50 de numere
model_w2v = Word2Vec(sentences=texte_tokenizate, vector_size=50, window=5, min_count=1, workers=4)

# 3. Funcție pentru a obține un singur vector per recenzie (media cuvintelor)
def obtine_vector_document(tokens, model):
    vectori = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectori) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectori, axis=0)

# 4. Generăm matricea finală pentru Word2Vec
X_w2v = np.array([obtine_vector_document(tokens, model_w2v) for tokens in texte_tokenizate])

df_w2v = pd.DataFrame(X_w2v, columns=[f'W2V_Dim_{i+1}' for i in range(50)])
print(f"Forma matricii Word2Vec: {df_w2v.shape} (texte x dimensiuni vectoriale)")
display(df_w2v.head(3))

=== 1. BAG OF WORDS (BoW) ===
Forma matricii BoW: (207, 474) (texte x cuvinte)


,abundant,ac,access,actually,added,adjust,adult,advertised,advised,agreed,...,whomever,wifi,window,winter,work,worked,working,workout,year,yr
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0



=== 2. TF-IDF ===
Forma matricii TF-IDF: (207, 474) (texte x cuvinte)


,abundant,ac,access,actually,added,adjust,adult,advertised,advised,agreed,...,whomever,wifi,window,winter,work,worked,working,workout,year,yr
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.644209,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0


=== 3. WORD2VEC ===
Forma matricii Word2Vec: (207, 50) (texte x dimensiuni vectoriale)


,W2V_Dim_1,W2V_Dim_2,W2V_Dim_3,W2V_Dim_4,W2V_Dim_5,W2V_Dim_6,W2V_Dim_7,W2V_Dim_8,W2V_Dim_9,W2V_Dim_10,...,W2V_Dim_41,W2V_Dim_42,W2V_Dim_43,W2V_Dim_44,W2V_Dim_45,W2V_Dim_46,W2V_Dim_47,W2V_Dim_48,W2V_Dim_49,W2V_Dim_50
0,-0.004251,-0.000286,-0.003632,0.002858,-0.002029,-0.005326,0.007051,0.001408,-0.010601,0.000010,...,-0.002804,0.002334,-0.002274,-0.006670,0.005939,-0.000839,-0.004065,-0.000624,0.009496,0.005722
1,0.004191,0.002078,0.003936,0.018239,0.000527,0.000017,0.009686,0.002219,0.005646,-0.000564,...,-0.000761,0.005861,-0.017073,0.008682,0.005210,-0.000478,-0.015891,0.001187,0.006128,0.012264
2,-0.005144,-0.004915,-0.019258,-0.015274,-0.002067,0.008480,0.006318,0.012095,0.013607,-0.005500,...,-0.008234,0.000182,-0.009530,0.006253,0.000785,-0.010011,-0.002680,-0.000246,0.003014,0.003655


In [8]:
print("=== 4. ALTE CARACTERISTICI (Stilometrice / Lexicale) ===")

def extrage_caracteristici_manuale(text):
    caracteristici = {}

    # 1. Lungimea textului (nr. de caractere)
    caracteristici['Nr_Caractere'] = len(text)

    # 2. Numărul de cuvinte
    cuvinte = word_tokenize(text)
    caracteristici['Nr_Cuvinte'] = len(cuvinte)

    # 3. Lungimea medie a cuvintelor
    if len(cuvinte) > 0:
        caracteristici['Lungime_Medie_Cuv'] = sum(len(c) for c in cuvinte) / len(cuvinte)
    else:
        caracteristici['Lungime_Medie_Cuv'] = 0

    # 4. Semne de punctuație (frecvența lor)
    caracteristici['Nr_Punctuatie'] = sum(1 for char in text if char in string.punctuation)

    # 5. Rata de majuscule (indică de obicei furie/entuziasm extrem)
    nr_litere = sum(1 for char in text if char.isalpha())
    nr_majuscule = sum(1 for char in text if char.isupper())
    caracteristici['Rata_Majuscule'] = (nr_majuscule / nr_litere) if nr_litere > 0 else 0

    return caracteristici

text_brut = df_curat['Text'].astype(str).tolist()
lista_caracteristici_noi = [extrage_caracteristici_manuale(text) for text in text_brut]

# Creăm dataframe-ul
df_alte_caracteristici = pd.DataFrame(lista_caracteristici_noi)
X_alte_caracteristici = df_alte_caracteristici.values # Matricea numpy pentru antrenare

print(f"Forma matricii 'Alte Caracteristici': {df_alte_caracteristici.shape} (texte x funcții custom)")
display(df_alte_caracteristici.head(3))

=== 4. ALTE CARACTERISTICI (Stilometrice / Lexicale) ===
Forma matricii 'Alte Caracteristici': (207, 5) (texte x funcții custom)


,Nr_Caractere,Nr_Cuvinte,Lungime_Medie_Cuv,Nr_Punctuatie,Rata_Majuscule
0,54,11,4.181818,2,0.022727
1,23,6,3.166667,1,0.055556
2,26,4,6.000000,1,0.043478


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
import pandas as pd

# 1. Pregătim etichetele (y)
y_brut = df_curat[COLOANA_ETICHETA].astype(str).str.lower().tolist()

encoder = LabelEncoder()
y = encoder.fit_transform(y_brut)

matrici_caracteristici = {
    "Bag of Words (BoW)": X_bow.toarray(),
    "TF-IDF": X_tfidf.toarray(),
    "Word2Vec": X_w2v,
    "Caracteristici Manuale": X_alte_caracteristici
}

rezultate_acuratețe = {}
modele_salvate = {}

for nume_metoda, X_curent in matrici_caracteristici.items():
    # Împărțim datele folosind STRATIFY pentru a evita dezechilibrele fatale
    X_train, X_test, y_train, y_test = train_test_split(
        X_curent, y, test_size=0.2, random_state=42)

    # --- PAS NOU ȘI CRUCIAL: SCALAREA DATELOR ---
    # StandardScaler aduce toate numerele la o medie de 0, eliminând diferențele uriașe de magnitudine
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    # ---------------------------------------------

    # Creștem max_iter la 1000 pentru a-i da mai mult timp să învețe
    model_ann = MLPClassifier(hidden_layer_sizes=(50, 20),
                              activation='relu',
                              solver='adam',
                              max_iter=1000,
                              early_stopping=True,
                              random_state=42)

    # Antrenăm modelul pe datele SCALATE
    model_ann.fit(X_train_scaled, y_train)

    # Testăm modelul
    predictii = model_ann.predict(X_test_scaled)
    acuratete = accuracy_score(y_test, predictii)

    rezultate_acuratețe[nume_metoda] = acuratete
    modele_salvate[nume_metoda] = model_ann

# Afișăm clasamentul
df_comparativ = pd.DataFrame(list(rezultate_acuratețe.items()), columns=['Metodă Extragere', 'Acuratețe ANN'])
df_comparativ['Acuratețe ANN'] = (df_comparativ['Acuratețe ANN'] * 100).round(2).astype(str) + '%'
df_comparativ = df_comparativ.sort_values(by='Acuratețe ANN', ascending=False).reset_index(drop=True)

display(df_comparativ)

,Metodă Extragere,Acuratețe ANN
0,TF-IDF,76.19%
1,Bag of Words (BoW),73.81%
2,Caracteristici Manuale,71.43%
3,Word2Vec,59.52%


In [11]:
from azure.core.credentials import AzureKeyCredential
from azure.ai.textanalytics import TextAnalyticsClient

print("=== PREZICEREA MESAJULUI (Comparație: Azure vs ANN cu Tool) ===\n")

mesaj_bicicleta = "By choosing a bike over a car, I’m reducing my environmental footprint. Cycling promotes eco-friendly transportation, and I’m proud to be part of that movement."
print(f"Mesaj testat:\n'{mesaj_bicicleta}'\n")

# ==========================================
# 1. PREDICȚIA FOLOSIND AZURE API (Subpunct 1)
# ==========================================
try:
    credential = AzureKeyCredential(AZURE_KEY)
    client = TextAnalyticsClient(endpoint=AZURE_ENDPOINT, credential=credential)
    response_azure = client.analyze_sentiment(documents=[mesaj_bicicleta])[0]
    sentiment_azure = response_azure.sentiment.upper()
except Exception as e:
    sentiment_azure = "EROARE (Verifică dacă cheia Azure este setată corect)"

print(f"-> Sentiment stabilit de AZURE API:      {sentiment_azure}")


# ==========================================
# 2. PREDICȚIA FOLOSIND ANN TOOL (Subpunct 3)
# ==========================================
# Transformăm mesajul folosind vocabularul învățat
vector_mesaj_tfidf = vectorizer_tfidf.transform([mesaj_bicicleta]).toarray()

# Extragem modelul TF-IDF deja antrenat din dicționar
model_tool_tfidf = modele_salvate["TF-IDF"]

# Refacem corect StandardScaler-ul DOAR pentru TF-IDF
X_train_tfidf, _, _, _ = train_test_split(
    matrici_caracteristici["TF-IDF"], y, test_size=0.2, random_state=42, stratify=y)

scaler_tool = StandardScaler()
scaler_tool.fit(X_train_tfidf)

# Scalam mesajul
vector_mesaj_scalat = scaler_tool.transform(vector_mesaj_tfidf)

# Facem predicția
predictie_tool_num = model_tool_tfidf.predict(vector_mesaj_scalat)[0]
sentiment_tool = encoder.inverse_transform([predictie_tool_num])[0]

print(f"-> Sentiment stabilit de ANN Tool local: {sentiment_tool.upper()}")

=== PREZICEREA MESAJULUI (Comparație: Azure vs ANN cu Tool) ===

Mesaj testat:
'By choosing a bike over a car, I’m reducing my environmental footprint. Cycling promotes eco-friendly transportation, and I’m proud to be part of that movement.'

-> Sentiment stabilit de AZURE API:      POSITIVE
-> Sentiment stabilit de ANN Tool local: NEGATIVE


In [12]:
import numpy as np

print("=== SUBPUNCTUL 4: ANN CU COD PROPRIU (De la zero) ===\n")

class ReteaNeuronalaProprie:
    def __init__(self, input_size, hidden_size, output_size=1, learning_rate=0.01):
        self.lr = learning_rate
        # Inițializăm greutățile și bias-urile cu valori aleatorii mici
        self.W1 = np.random.randn(input_size, hidden_size) * 0.1
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * 0.1
        self.b2 = np.zeros((1, output_size))

    def sigmoid(self, z):
        # Funcția de activare (clip pentru a evita erori matematice de depășire)
        z = np.clip(z, -250, 250)
        return 1.0 / (1.0 + np.exp(-z))

    def derivata_sigmoid(self, z):
        s = self.sigmoid(z)
        return s * (1.0 - s)

    def forward(self, X):
        # Propagarea înainte (cum "gândește" rețeaua)
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = self.sigmoid(self.z1)
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = self.sigmoid(self.z2)
        return self.a2

    def backward(self, X, y, output):
        # Propagarea înapoi (cum "învață" rețeaua din greșeli)
        m = X.shape[0]
        y = y.reshape(-1, 1)

        dz2 = output - y
        dW2 = (1 / m) * np.dot(self.a1.T, dz2)
        db2 = (1 / m) * np.sum(dz2, axis=0, keepdims=True)

        dz1 = np.dot(dz2, self.W2.T) * self.derivata_sigmoid(self.z1)
        dW1 = (1 / m) * np.dot(X.T, dz1)
        db1 = (1 / m) * np.sum(dz1, axis=0, keepdims=True)

        # Ajustăm greutățile
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2

    def fit(self, X, y, epoci=1500):
        for _ in range(epoci):
            output = self.forward(X)
            self.backward(X, y, output)

    def predict(self, X):
        probabilitati = self.forward(X)
        # 1 dacă probabilitatea > 50%, altfel 0
        return (probabilitati >= 0.5).astype(int).flatten()

# 1. Alegem cea mai bună caracteristică de mai devreme (BoW a ieșit bine!)
X_cel_mai_bun = matrici_caracteristici["Bag of Words (BoW)"]
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X_cel_mai_bun, y, test_size=0.2, random_state=42, stratify=y)

scaler_final = StandardScaler()
X_train_scaled_final = scaler_final.fit_transform(X_train_final)
X_test_scaled_final = scaler_final.transform(X_test_final)

# 2. Creăm și antrenăm "creierul" nostru manual
ann_propriu = ReteaNeuronalaProprie(input_size=X_train_scaled_final.shape[1], hidden_size=20, learning_rate=0.1)

print("Antrenăm rețeaua construită de la zero... (durează câteva secunde)")
ann_propriu.fit(X_train_scaled_final, y_train_final, epoci=2000)

# 3. Calculăm acuratețea
predictii_proprii = ann_propriu.predict(X_test_scaled_final)
print(f"Acuratețe ANN (Cod Propriu): {accuracy_score(y_test_final, predictii_proprii) * 100:.2f}%\n")

# 4. REZOLVAREA CERINȚEI (Mesajul cu bicicleta)
mesaj_laborator = "By choosing a bike over a car, I’m reducing my environmental footprint. Cycling promotes eco-friendly transportation, and I’m proud to be part of that movement."

# Transformăm textul folosind vocabularul BoW (vectorizer_bow)
vector_mesaj = vectorizer_bow.transform([mesaj_laborator]).toarray()
vector_mesaj_scalat = scaler_final.transform(vector_mesaj)

# Facem predicția finală
predictie_numar = ann_propriu.predict(vector_mesaj_scalat)[0]
sentiment_final = encoder.inverse_transform([predictie_numar])[0]

print(f"Mesaj test: '{mesaj_laborator}'")
print(f"-> Sentiment stabilit (Cod Propriu): {sentiment_final.upper()}")

=== SUBPUNCTUL 4: ANN CU COD PROPRIU (De la zero) ===

Antrenăm rețeaua construită de la zero... (durează câteva secunde)
Acuratețe ANN (Cod Propriu): 76.19%

Mesaj test: 'By choosing a bike over a car, I’m reducing my environmental footprint. Cycling promotes eco-friendly transportation, and I’m proud to be part of that movement.'
-> Sentiment stabilit (Cod Propriu): NEGATIVE
